# Final reproducible competition solution — team 4BD478E9

This notebook rebuilds leakage-safe transaction features from the raw competition files, evaluates the selected model on the fixed expanding chronological OOF folds, refits on all labeled signals, and writes a validated test prediction file. The selected predictor is shallow XGBoost, based on the controlled fixed-fold sprint; the extra trees and histogram-boosting baselines remain documented in `experiments/controlled_optimization/`.

**Run from a clean kernel:** use *Restart Kernel and Run All*. The notebook discovers the project root from this file or the current working directory; no variables from another notebook are required. Its optional cache is version-checked against raw file fingerprints. Set `FORCE_REBUILD_FEATURES = True` in the configuration cell to rebuild from raw data.

## 1. Environment and imports

In [ ]:
from __future__ import annotations
import gc
import hashlib
import json
import os
import platform
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import sklearn
import xgboost
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

print(f"Python {platform.python_version()} | pandas {pd.__version__} | NumPy {np.__version__} | scikit-learn {sklearn.__version__} | XGBoost {xgboost.__version__}")


## 2. Configuration

In [ ]:
def discover_project_root(start=None) -> Path:
    candidates = [Path.cwd(), Path.cwd().resolve(), Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd()]
    if start is not None:
        candidates.insert(0, start)
    for candidate in candidates:
        for parent in (candidate, *candidate.parents):
            if (parent / "data" / "raw" / "train_signals.csv").exists():
                return parent
    raise FileNotFoundError("Project root not found. Start Jupyter from the project directory or a child directory.")

PROJECT_ROOT = discover_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "raw"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "final_solution"
PREDICTION_DIR = PROJECT_ROOT / "artifacts" / "predictions"
FEATURE_CACHE_DIR = ARTIFACT_DIR / "features"
SEED = 42
N_SPLITS = 5
WARMUP_DATE_FRACTION = 1 / (N_SPLITS + 1)
FORCE_REBUILD_FEATURES = False  # Set True to rebuild caches from raw files.
USE_FEATURE_CACHE = True

SIGNAL_ID = "signal_id"
SIGNAL_DATE = "signal_sanasi"
TARGET = "eskalatsiya"
TX_DATE = "tranzaksiya_vaqti"
TX_DIRECTION = "kirim_chiqim"
TX_TYPE = "tranzaksiya_turi"
TX_AMOUNT = "miqdor_indeksi"

TRAIN_SIGNALS_PATH = DATA_DIR / "train_signals.csv"
TEST_SIGNALS_PATH = DATA_DIR / "test_signals.csv"
TRAIN_TX_PATH = DATA_DIR / "train_transactions.parquet"
TEST_TX_PATH = DATA_DIR / "test_transactions.parquet"
SAMPLE_SUBMISSION_PATH = DATA_DIR / "sample_submission (3).csv"
SUBMISSION_PATH = PREDICTION_DIR / "final_submission.csv"

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
FEATURE_CACHE_DIR.mkdir(parents=True, exist_ok=True)
PREDICTION_DIR.mkdir(parents=True, exist_ok=True)
np.random.seed(SEED)
print("Project root discovered:", PROJECT_ROOT.name)
print("Seed:", SEED, "| expanding validation folds:", N_SPLITS, "| warm-up date fraction:", WARMUP_DATE_FRACTION)


## 3. Load raw files

In [ ]:
train_signals = pd.read_csv(TRAIN_SIGNALS_PATH, parse_dates=[SIGNAL_DATE])
test_signals = pd.read_csv(TEST_SIGNALS_PATH, parse_dates=[SIGNAL_DATE])
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

# Read Parquet metadata here; feature generation reads one transaction table at a time to limit peak memory.
train_tx_metadata = pq.ParquetFile(TRAIN_TX_PATH).metadata
test_tx_metadata = pq.ParquetFile(TEST_TX_PATH).metadata

assert train_signals[TARGET].notna().all()
assert set(train_signals[TARGET].unique()).issubset({0, 1})
assert train_signals[SIGNAL_ID].is_unique and test_signals[SIGNAL_ID].is_unique
assert set(train_signals.columns) == {SIGNAL_ID, SIGNAL_DATE, TARGET}
assert set(test_signals.columns) == {SIGNAL_ID, SIGNAL_DATE}

print(f"Train signals: {len(train_signals):,} | test signals: {len(test_signals):,}")
print(f"Train transactions: {train_tx_metadata.num_rows:,} | test transactions: {test_tx_metadata.num_rows:,}")
print("Training date range:", train_signals[SIGNAL_DATE].min(), "to", train_signals[SIGNAL_DATE].max())
print("Test date range:    ", test_signals[SIGNAL_DATE].min(), "to", test_signals[SIGNAL_DATE].max())
print("Training target counts:\n", train_signals[TARGET].value_counts().sort_index())


## 4. Data audit summary

In [ ]:
train_schema = pq.read_schema(TRAIN_TX_PATH).names
test_schema = pq.read_schema(TEST_TX_PATH).names
audit_rows = []
for split_name, signals, tx_path, tx_meta, tx_schema in [
    ("train", train_signals, TRAIN_TX_PATH, train_tx_metadata, train_schema),
    ("test", test_signals, TEST_TX_PATH, test_tx_metadata, test_schema),
]:
    audit_rows.append({
        "split": split_name,
        "signals": len(signals),
        "unique_signal_ids": signals[SIGNAL_ID].nunique(),
        "transaction_rows": tx_meta.num_rows,
        "transaction_schema": ", ".join(tx_schema),
        "signal_date_min": signals[SIGNAL_DATE].min().date(),
        "signal_date_max": signals[SIGNAL_DATE].max().date(),
        "target_prevalence": float(signals[TARGET].mean()) if TARGET in signals else np.nan,
    })
display(pd.DataFrame(audit_rows))
print("Signal ID overlap across train and test:", len(set(train_signals[SIGNAL_ID]) & set(test_signals[SIGNAL_ID])))


## 5. Temporal rule

In [ ]:
# Only transactions dated on or before their linked signal timestamp may contribute features.
# Audit this rule directly from the raw training transaction table before building/cache-loading features.
train_tx_audit = pd.read_parquet(TRAIN_TX_PATH, columns=[SIGNAL_ID, TX_DATE])
train_tx_audit[TX_DATE] = pd.to_datetime(train_tx_audit[TX_DATE], errors="coerce")
train_tx_audit = train_tx_audit.merge(train_signals[[SIGNAL_ID, SIGNAL_DATE]], on=SIGNAL_ID, how="left", validate="many_to_one")
missing_link = train_tx_audit[SIGNAL_DATE].isna().sum()
missing_tx_date = train_tx_audit[TX_DATE].isna().sum()
future_mask = train_tx_audit[TX_DATE] > train_tx_audit[SIGNAL_DATE]
future_count = int(future_mask.sum())
future_signal_count = int(train_tx_audit.loc[future_mask, SIGNAL_ID].nunique())
dated_linked_count = int((train_tx_audit[SIGNAL_DATE].notna() & train_tx_audit[TX_DATE].notna()).sum())
print(f"Unmatched transaction IDs: {missing_link:,} | missing transaction dates: {missing_tx_date:,}")
print(f"Transactions after signal time: {future_count:,} / {dated_linked_count:,} ({future_count / max(dated_linked_count, 1):.3%})")
print(f"Signals with at least one future transaction: {future_signal_count:,} / {len(train_signals):,}")
del train_tx_audit
gc.collect()


## 6. Feature generation (with optional version-checked cache)

Features are generated from the actual raw columns: `kirim_chiqim`, `tranzaksiya_turi`, and `miqdor_indeksi`. The cache is accepted only when source file size and modification time match its manifest. Set `FORCE_REBUILD_FEATURES = True` above to bypass it.

In [ ]:
def source_fingerprint(paths: list[Path]) -> dict:
    return {p.name: {"size": p.stat().st_size, "mtime_ns": p.stat().st_mtime_ns} for p in paths}

FEATURE_MANIFEST_PATH = FEATURE_CACHE_DIR / "manifest.json"
TRAIN_FEATURES_PATH = FEATURE_CACHE_DIR / "train_features.parquet"
TEST_FEATURES_PATH = FEATURE_CACHE_DIR / "test_features.parquet"
FEATURE_VERSION = "1.0.1"
SOURCE_FINGERPRINT = source_fingerprint([TRAIN_SIGNALS_PATH, TEST_SIGNALS_PATH, TRAIN_TX_PATH, TEST_TX_PATH])


def generate_signal_features(signals: pd.DataFrame, transactions_path: Path, split_name: str) -> pd.DataFrame:
    # Aggregate transaction behavior as-of each signal, then release transaction memory.
    tx = pd.read_parquet(transactions_path, columns=[SIGNAL_ID, TX_DATE, TX_DIRECTION, TX_TYPE, TX_AMOUNT])
    tx[TX_DATE] = pd.to_datetime(tx[TX_DATE], errors="coerce")
    tx[TX_AMOUNT] = pd.to_numeric(tx[TX_AMOUNT], errors="coerce")
    linked = tx.merge(signals[[SIGNAL_ID, SIGNAL_DATE]], on=SIGNAL_ID, how="left", validate="many_to_one")
    if linked[SIGNAL_DATE].isna().any():
        raise ValueError(f"{split_name}: transactions reference unknown signal IDs")
    linked = linked.loc[linked[TX_DATE].notna() & (linked[TX_DATE] <= linked[SIGNAL_DATE])].copy()
    linked["_calendar_day"] = linked[TX_DATE].dt.normalize()

    grouped = linked.groupby(SIGNAL_ID, sort=False, observed=True)
    out = grouped.agg(
        transaction_count=(TX_DATE, "size"),
        active_days=("_calendar_day", "nunique"),
        first_transaction=(TX_DATE, "min"),
        last_transaction=(TX_DATE, "max"),
        amount_sum=(TX_AMOUNT, "sum"),
        amount_mean=(TX_AMOUNT, "mean"),
        amount_std=(TX_AMOUNT, "std"),
        amount_median=(TX_AMOUNT, "median"),
        amount_min=(TX_AMOUNT, "min"),
        amount_max=(TX_AMOUNT, "max"),
    )
    amount_q = grouped[TX_AMOUNT].quantile([0.75, 0.90]).unstack()
    amount_q.columns = ["amount_q75", "amount_q90"]
    out = out.join(amount_q)

    direction_counts = pd.crosstab(linked[SIGNAL_ID], linked[TX_DIRECTION])
    direction_counts = direction_counts.reindex(columns=["kirim", "chiqim"], fill_value=0)
    direction_counts.columns = ["incoming_count", "outgoing_count"]
    out = out.join(direction_counts)
    type_values = ["karta", "bank_otkazmasi", "naqd", "xalqaro"]
    type_counts = pd.crosstab(linked[SIGNAL_ID], linked[TX_TYPE]).reindex(columns=type_values, fill_value=0)
    type_counts.columns = [f"type_{name}_count" for name in type_values]
    out = out.join(type_counts)

    # Fixed short windows include the signal date and the preceding (window - 1) calendar days.
    signal_dates = signals.set_index(SIGNAL_ID)[SIGNAL_DATE]
    linked_signal_dates = linked[SIGNAL_ID].map(signal_dates)
    for days in (7, 30, 90):
        recent = linked.loc[linked[TX_DATE] >= linked_signal_dates - pd.Timedelta(days=days - 1)]
        recent_grouped = recent.groupby(SIGNAL_ID, sort=False, observed=True)
        window = pd.DataFrame({
            f"transaction_count_{days}d": recent_grouped.size(),
            f"amount_sum_{days}d": recent_grouped[TX_AMOUNT].sum(),
        })
        out = out.join(window)

    signal_dates = signals.set_index(SIGNAL_ID)[SIGNAL_DATE]
    out["incoming_share"] = out["incoming_count"] / out["transaction_count"]
    out["outgoing_share"] = out["outgoing_count"] / out["transaction_count"]
    out["days_since_last_transaction"] = (signal_dates - out["last_transaction"]).dt.total_seconds() / 86400
    out["history_span_days"] = (out["last_transaction"] - out["first_transaction"]).dt.total_seconds() / 86400
    out["signal_month"] = signal_dates.dt.month
    out["signal_day_of_week"] = signal_dates.dt.dayofweek
    out = out.drop(columns=["first_transaction", "last_transaction"])

    # Return rows in signal-file order and retain no post-signal values.
    out = signals[[SIGNAL_ID]].merge(out.reset_index(), on=SIGNAL_ID, how="left", sort=False, validate="one_to_one")
    numeric_features = [c for c in out.columns if c != SIGNAL_ID and pd.api.types.is_numeric_dtype(out[c])]
    out.loc[:, numeric_features] = out[numeric_features].replace([np.inf, -np.inf], np.nan)
    count_cols = [c for c in numeric_features if "count" in c or c == "active_days"]
    out.loc[:, count_cols] = out[count_cols].fillna(0)
    out["days_since_last_transaction"] = out["days_since_last_transaction"].fillna(9999)
    out["history_span_days"] = out["history_span_days"].fillna(0)
    out.loc[:, numeric_features] = out[numeric_features].fillna(0)
    if out[SIGNAL_ID].duplicated().any() or len(out) != len(signals):
        raise AssertionError(f"{split_name}: expected exactly one feature row per signal")
    print(f"{split_name}: retained {len(linked):,} as-of transactions; feature matrix {out.shape}")
    del tx, linked, grouped
    gc.collect()
    return out


cache_is_valid = False
if USE_FEATURE_CACHE and not FORCE_REBUILD_FEATURES and FEATURE_MANIFEST_PATH.exists() and TRAIN_FEATURES_PATH.exists() and TEST_FEATURES_PATH.exists():
    manifest = json.loads(FEATURE_MANIFEST_PATH.read_text(encoding="utf-8"))
    cache_is_valid = manifest.get("feature_version") == FEATURE_VERSION and manifest.get("source_fingerprint") == SOURCE_FINGERPRINT

if cache_is_valid:
    train_features = pd.read_parquet(TRAIN_FEATURES_PATH)
    test_features = pd.read_parquet(TEST_FEATURES_PATH)
    print("Loaded feature cache after source fingerprint validation.")
else:
    train_features = generate_signal_features(train_signals, TRAIN_TX_PATH, "train")
    test_features = generate_signal_features(test_signals, TEST_TX_PATH, "test")
    train_features.to_parquet(TRAIN_FEATURES_PATH, index=False)
    test_features.to_parquet(TEST_FEATURES_PATH, index=False)
    FEATURE_MANIFEST_PATH.write_text(json.dumps({"feature_version": FEATURE_VERSION, "source_fingerprint": SOURCE_FINGERPRINT, "feature_columns": train_features.columns.tolist(), "seed": SEED}, indent=2), encoding="utf-8")
    print("Rebuilt and saved feature cache.")

assert train_features[SIGNAL_ID].tolist() == train_signals[SIGNAL_ID].tolist()
assert test_features[SIGNAL_ID].tolist() == test_signals[SIGNAL_ID].tolist()
feature_columns = [c for c in train_features.columns if c != SIGNAL_ID]
assert feature_columns == [c for c in test_features.columns if c != SIGNAL_ID]
X = train_features[feature_columns].astype("float32")
X_test = test_features[feature_columns].astype("float32")
y = train_signals[TARGET].astype("int8").reset_index(drop=True)
print(f"Features: {len(feature_columns)} | train: {X.shape} | test: {X_test.shape}")


## 7. Expanding chronological validation

In [ ]:
# The earliest date block is warm-up only. Each scored block is later than all rows used to train that fold.
unique_dates = np.array(sorted(train_signals[SIGNAL_DATE].dropna().unique()))
date_blocks = np.array_split(unique_dates, N_SPLITS + 1)
folds = []
date_values = train_signals[SIGNAL_DATE].to_numpy()
for fold_number, validation_dates in enumerate(date_blocks[1:], start=1):
    if len(validation_dates) == 0:
        continue
    first_validation_date = validation_dates.min()
    train_idx = np.flatnonzero(date_values < first_validation_date)
    last_validation_date = validation_dates.max()
    validation_idx = np.flatnonzero((date_values >= first_validation_date) & (date_values <= last_validation_date))
    if len(train_idx) == 0 or len(validation_idx) == 0:
        continue
    folds.append({"fold": fold_number, "train_idx": train_idx, "validation_idx": validation_idx,
                  "train_max_date": pd.Timestamp(date_values[train_idx].max()),
                  "validation_min_date": pd.Timestamp(date_values[validation_idx].min()),
                  "validation_max_date": pd.Timestamp(date_values[validation_idx].max())})
    assert folds[-1]["train_max_date"] < folds[-1]["validation_min_date"]

fold_summary = pd.DataFrame([{
    "fold": f["fold"], "train_rows": len(f["train_idx"]), "validation_rows": len(f["validation_idx"]),
    "train_through": f["train_max_date"].date(), "validation_from": f["validation_min_date"].date(),
    "validation_to": f["validation_max_date"].date(),
    "validation_positive_rate": float(y.iloc[f["validation_idx"]].mean()),
} for f in folds])
assert len(folds) == N_SPLITS, f"Expected {N_SPLITS} folds; got {len(folds)}"
display(fold_summary)
fold_definitions = [{
    "fold": f["fold"], "train_idx": f["train_idx"].tolist(), "validation_idx": f["validation_idx"].tolist(),
    "train_max_date": f["train_max_date"].isoformat(), "validation_min_date": f["validation_min_date"].isoformat(),
    "validation_max_date": f["validation_max_date"].isoformat(),
} for f in folds]
(ARTIFACT_DIR / "fold_definitions.json").write_text(json.dumps(fold_definitions, indent=2), encoding="utf-8")


## 8. Model training and 9. OOF evaluation

In [ ]:
MODEL_FACTORIES = {
    "xgboost_shallow": lambda: XGBClassifier(
        n_estimators=500, max_depth=2, learning_rate=0.025, min_child_weight=15,
        subsample=0.8, colsample_bytree=0.8, reg_lambda=5.0,
        eval_metric="logloss", n_jobs=-1, random_state=SEED,
    ),
}

# OOF predictions are generated on exactly the stored expanding chronological folds.
oof_predictions = pd.DataFrame(index=np.arange(len(y)))
oof_predictions[TARGET] = y.to_numpy()
oof_predictions["signal_date"] = train_signals[SIGNAL_DATE].to_numpy()
fold_metrics = []
for model_name, make_model in MODEL_FACTORIES.items():
    print(f"Training {model_name} across {len(folds)} forward-only folds")
    oof_column = np.full(len(y), np.nan, dtype="float64")
    for fold in folds:
        model = make_model()
        train_idx, validation_idx = fold["train_idx"], fold["validation_idx"]
        started = time.time()
        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        positive_column = list(model.classes_).index(1)
        probabilities = model.predict_proba(X.iloc[validation_idx])[:, positive_column]
        oof_column[validation_idx] = probabilities
        score = roc_auc_score(y.iloc[validation_idx], probabilities)
        fold_metrics.append({"model": model_name, "fold": fold["fold"], "roc_auc": score,
                             "train_rows": len(train_idx), "validation_rows": len(validation_idx),
                             "fit_seconds": time.time() - started})
        print(f"  fold {fold['fold']}: ROC-AUC={score:.5f} ({time.time() - started:.1f}s)")
    oof_predictions[model_name] = oof_column

oof_mask = oof_predictions[list(MODEL_FACTORIES)].notna().all(axis=1)
assert oof_mask.any() and oof_predictions.loc[oof_mask, TARGET].nunique() == 2
oof_summary = []
for model_name in MODEL_FACTORIES:
    oof_summary.append({"model": model_name, "oof_roc_auc": roc_auc_score(oof_predictions.loc[oof_mask, TARGET], oof_predictions.loc[oof_mask, model_name])})
oof_summary = pd.DataFrame(oof_summary).sort_values("oof_roc_auc", ascending=False)
display(oof_summary)
display(pd.DataFrame(fold_metrics))
print(f"OOF coverage: {int(oof_mask.sum()):,}/{len(y):,} ({oof_mask.mean():.1%}); initial warm-up rows are intentionally unscored.")
oof_predictions.loc[oof_mask].to_parquet(ARTIFACT_DIR / "oof_predictions.parquet", index=False)
oof_summary.to_csv(ARTIFACT_DIR / "oof_metrics.csv", index=False)
pd.DataFrame(fold_metrics).to_csv(ARTIFACT_DIR / "fold_metrics.csv", index=False)


## 10. Ensemble construction and final refit

The sprint compared a fixed set of forward-only blends. They did not beat the shallow XGBoost challenger alone, so the final predictor uses a single model with weight 1.0. No test-set labels or leaderboard feedback were used to choose it.

In [ ]:
# The controlled sprint found no forward-only blend that beat this candidate alone.
ensemble_weights = {"xgboost_shallow": 1.0}
print("Final predictor weights:", ensemble_weights)

final_models = {}
test_prediction_frame = pd.DataFrame(index=np.arange(len(test_signals)))
for model_name, make_model in MODEL_FACTORIES.items():
    final_model = make_model()
    final_model.fit(X, y)
    positive_column = list(final_model.classes_).index(1)
    test_prediction_frame[model_name] = final_model.predict_proba(X_test)[:, positive_column]
    final_models[model_name] = final_model

test_prediction_frame["ehtimollik"] = sum(test_prediction_frame[name] * weight for name, weight in ensemble_weights.items())
print(test_prediction_frame.describe())


## 11. Final test predictions and 12. Submission validation

In [ ]:
submission = pd.DataFrame({
    SIGNAL_ID: test_signals[SIGNAL_ID].to_numpy(),
    "ehtimollik": test_prediction_frame["ehtimollik"].to_numpy(dtype="float64"),
})

assert submission.columns.tolist() == sample_submission.columns.tolist(), (submission.columns.tolist(), sample_submission.columns.tolist())
assert len(submission) == len(test_signals) == len(sample_submission)
assert submission[SIGNAL_ID].equals(test_signals[SIGNAL_ID].reset_index(drop=True))
assert submission[SIGNAL_ID].is_unique
assert np.isfinite(submission["ehtimollik"]).all()
assert submission["ehtimollik"].between(0.0, 1.0).all()
assert not submission.isna().any().any()

submission.to_csv(SUBMISSION_PATH, index=False)
print("Submission checks passed:", len(submission), "rows; columns:", submission.columns.tolist())
display(submission.head())


## 13. Saved output paths

In [ ]:
print("Competition submission:", SUBMISSION_PATH.relative_to(PROJECT_ROOT))
print("Feature cache:", FEATURE_CACHE_DIR.relative_to(PROJECT_ROOT))
print("OOF predictions:", (ARTIFACT_DIR / 'oof_predictions.parquet').relative_to(PROJECT_ROOT))
print("OOF metrics:", (ARTIFACT_DIR / 'oof_metrics.csv').relative_to(PROJECT_ROOT))
print("Fold metrics:", (ARTIFACT_DIR / 'fold_metrics.csv').relative_to(PROJECT_ROOT))


### Reproduction notes

- Install repository dependencies with `python -m pip install -r requirements.txt`; run this notebook with **Restart Kernel and Run All**.
- The selected model is fixed in this notebook: XGBoost with 500 trees, depth 2, learning rate 0.025, minimum child weight 15, subsample 0.8, column sample 0.8, L2 regularization 5, and seed 42.
- Validation is the same five expanding chronological folds persisted in `artifacts/final_solution/fold_definitions.json`. A warm-up date block is excluded from OOF scoring.
- The first run reads the raw transaction Parquet files sequentially and writes compressed per-signal feature caches. Later runs reuse them only when source file size and modification times match the manifest.
- Delete `artifacts/final_solution/features/` or set `FORCE_REBUILD_FEATURES = True` to regenerate caches from raw data.
- Submission output is `artifacts/predictions/final_submission.csv`; create the team-named deliverable with `python -m src.submission.make_submission --team-id 4BD478E9` and validate it with `python -m src.submission.validate_submission --submission artifacts/predictions/team_4BD478E9.csv`.